In [35]:
import pandas as pd





In [36]:
df = pd.read_csv("C:\\Users\\dlhogan\\OneDrive - UW\\Documents\\GitHub\\Cascski-Alpine-Insights\\cascade-mountain-weather.github.io\\data\\forecasts\\Snoqualmie Pass.csv")
df['ValidTime'] = pd.to_datetime(df['ValidTime'], format="%Y%m%d%H")
df = df.set_index('ValidTime')

In [37]:
df

,APCP12hr_surface,APCP12hr_surface_1% level,APCP12hr_surface_10% level,APCP12hr_surface_15% level,APCP12hr_surface_20% level,APCP12hr_surface_25% level,APCP12hr_surface_30% level,APCP12hr_surface_35% level,APCP12hr_surface_40% level,APCP12hr_surface_45% level,...,WIND_10 m above ground_prob >15,WIND_10 m above ground_prob >17,WIND_10 m above ground_prob >24,WIND_10 m above ground_prob >3,WIND_10 m above ground_prob >32,WIND_10 m above ground_prob >5,WIND_10 m above ground_prob >8,WIND_30 m above ground,WIND_80 m above ground,WIND_surface - 610 m above ground
ValidTime,,,,,,,,,,,,,,,,,,,,,
2025-12-05 07:00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,NaN,NaN
2025-12-05 08:00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,NaN,NaN
2025-12-05 09:00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,NaN,NaN
2025-12-05 10:00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,NaN,NaN
2025-12-05 11:00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2025-12-16 05:00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2025-12-16 06:00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.6,2.0,NaN
2025-12-16 07:00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [30]:
dates = slice(df.index[0], df.index[0] + pd.Timedelta(hours=72))

In [32]:
min_snow_level = df['SNOWLVL_surface_50% level'].loc[dates].min()
max_snow_level = df['SNOWLVL_surface_50% level'].loc[dates].max()

In [34]:
min_snow_level, max_snow_level

(np.float64(1224.0), np.float64(2096.0))

In [15]:
import pandas as pd
import matplotlib.pyplot as plt
import os
import json
from datetime import datetime
sites = [
    "Mt. Baker Ski Area",
    "Washington Pass",
    "Stevens Pass",
    "TBLEW",
    "Snoqualmie Pass",
    "White Pass",
    "HURW1"
]
today = datetime.today().strftime('%Y-%m-%d')
# copy template 
with open(os.path.join(os.getcwd(), 'data', 'forecasts', f'eval_forecast_template.json'), 'r') as f:
        eval_forecast_template = json.load(f)
# rename to today's date
with open(os.path.join(os.getcwd(), 'data', 'forecasts', f'eval_forecast_{today}.json'), 'w') as f:
        json.dump(eval_forecast_template, f, indent=4)
for site in sites:
        working_dir = os.getcwd()
        col_df = pd.read_csv(os.path.join(working_dir, 'data', 'forecasts', 'nbm_cols.csv'))
        cols = ['ValidTime',
                'ASNOW72hr_surface_25% level',
                'ASNOW72hr_surface_50% level',
                'ASNOW72hr_surface_75% level', 
                "ASNOW72hr_surface",
                'ASNOW1hr_surface_25% level',
                'ASNOW1hr_surface_50% level',
                'ASNOW1hr_surface_75% level', 
                "ASNOW1hr_surface",
                'ASNOW6hr_surface_25% level',
                'ASNOW6hr_surface_50% level',
                'ASNOW6hr_surface_75% level', 
                "ASNOW6hr_surface",
                'ASNOW24hr_surface_25% level',
                'ASNOW24hr_surface_50% level',
                'ASNOW24hr_surface_75% level', 
                "ASNOW24hr_surface"]

        nbm_df = pd.read_csv(os.path.join(working_dir, 'data', 'forecasts', f'{site}.csv'))
        nbm_df = nbm_df[cols]
        nbm_df['ValidTime'] = pd.to_datetime(nbm_df['ValidTime'], format="%Y%m%d%H")
        nbm_df = nbm_df.set_index('ValidTime')

        if site == "HURW1":
                site = "Hurricane Ridge"
        elif site == "TBLEW":
                site = "Blewett Pass"
        dates = slice(nbm_df.index[0], nbm_df.index[0] + pd.Timedelta(hours=72))
        with open(os.path.join(working_dir, 'data', 'forecasts', f'eval_forecast_{today}.json'), 'r') as f:
                eval_forecast_data = json.load(f)
        eval_forecast_data['areas'][site]['nbm_forecast']['deterministic'] = round((nbm_df.loc[dates, 'ASNOW6hr_surface']*4*10).sum(),3)
        eval_forecast_data['areas'][site]['nbm_forecast']['ensemble_iqr_range'] = [round((nbm_df.loc[dates, 'ASNOW6hr_surface_25% level']*4*10).sum(),3),
                                                            round((nbm_df.loc[dates, 'ASNOW6hr_surface_75% level']*4*10).sum(),3)]
        # write to file
        with open(os.path.join(working_dir, 'data', 'forecasts', f'eval_forecast_{today}.json'), 'w') as f:
                json.dump(eval_forecast_data, f, indent=4)
